# DocuExtract — Fine-Tune LayoutLMv3 on SROIE (self-contained)

Trains receipt key-information extraction for **all four fields** (Company, Date, Address, Total).

**How to use:** Runtime → Change runtime type → **GPU (T4)**, then **Run all**. ~30–60 min. Produces `best_layoutlmv3_sroie.zip` to download.

This notebook is standalone — it does not clone any repo. The fix vs. the earlier version: text is tokenized **word-level** (not line-level) and a gap-tolerant tagger labels multi-word Company/Address spans, so the model actually learns them.


## 1. GPU check & dependencies


In [ ]:
!nvidia-smi
!pip -q install 'transformers>=4.40,<4.50' 'datasets>=2.18' seqeval evaluate torchvision pillow


## 2. Load the real SROIE dataset


In [ ]:
from datasets import load_dataset

dataset = load_dataset('jsdnrs/ICDAR2019-SROIE')
print(dataset)
print('fields:', dataset['train'].column_names)
print('example entities:', dataset['train'][0]['entities'])


## 3. Labels & BIO schema


In [ ]:
label_list = ['O',
    'B-COMPANY', 'I-COMPANY',
    'B-DATE', 'I-DATE',
    'B-ADDRESS', 'I-ADDRESS',
    'B-TOTAL', 'I-TOTAL']
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
FIELDS = ['company', 'date', 'address', 'total']
print(label2id)


## 4. Word-level tokens + robust BIO tagging (the fix)
Splits each OCR line into words (each inheriting its line box), normalizes boxes to [0,1000], and tags entity spans with gap-tolerant multi-word matching.


In [ ]:
import re

def normalize_bbox(box, w, h, scale=1000):
    x0, y0, x1, y1 = box
    if x0 > x1: x0, x1 = x1, x0
    if y0 > y1: y0, y1 = y1, y0
    nb = [int(round(scale * x0 / max(1, w))), int(round(scale * y0 / max(1, h))),
          int(round(scale * x1 / max(1, w))), int(round(scale * y1 / max(1, h)))]
    nb = [max(0, min(scale, v)) for v in nb]
    nb[2] = max(nb[0], nb[2]); nb[3] = max(nb[1], nb[3])
    return nb

def clean_text(t):
    return re.sub(r'[^\w\s]', '', str(t)).lower().strip()

def find_subsequence(seq, target):
    out = []
    if not target or len(target) > len(seq):
        return out
    n = len(target)
    for i in range(len(seq) - n + 1):
        if seq[i:i+n] == target:
            out.append((i, i+n))
    return out

def find_span(cleaned, val_tokens, occupied, gap=1):
    n = len(cleaned)
    for s, e in find_subsequence(cleaned, val_tokens):
        if not any(occupied[i] for i in range(s, e)):
            return (s, e)
    if len(val_tokens) == 1:
        tw = val_tokens[0]
        if len(tw) < 2:
            return None
        for i, c in enumerate(cleaned):
            if not occupied[i] and c and (tw == c or tw in c or c in tw):
                return (i, i+1)
        return None
    vs = set(val_tokens)
    matched = [i for i in range(n) if not occupied[i] and cleaned[i] in vs]
    if not matched:
        return None
    runs, cur = [], [matched[0]]
    for i in matched[1:]:
        if i - cur[-1] - 1 <= gap:
            cur.append(i)
        else:
            runs.append(cur); cur = [i]
    runs.append(cur)
    cov = lambda r: len({cleaned[i] for i in r} & vs)
    best = max(runs, key=cov)
    if cov(best) < max(2, (len(vs) + 1) // 2):
        return None
    s, e = best[0], best[-1] + 1
    if any(occupied[i] for i in range(s, e)):
        return None
    return (s, e)

def tag_tokens(tokens, gt):
    tags = ['O'] * len(tokens)
    cleaned = [clean_text(t) for t in tokens]
    occupied = [False] * len(tokens)
    ents = sorted([(k, gt[k]) for k in FIELDS if gt.get(k)], key=lambda x: len(x[1]), reverse=True)
    for key, val in ents:
        vt = [clean_text(w) for w in str(val).split() if clean_text(w)]
        if not vt:
            continue
        span = find_span(cleaned, vt, occupied)
        if span is None:
            continue
        s, e = span
        tags[s] = f'B-{key.upper()}'; occupied[s] = True
        for i in range(s+1, e):
            tags[i] = f'I-{key.upper()}'; occupied[i] = True
    return tags

def row_to_words_boxes(row):
    size = row.get('image_size') or {}
    w, h = size.get('width'), size.get('height')
    if not w or not h:
        w, h = row['image'].size
    toks, boxes = [], []
    for line, box in zip(row['words'], row['bboxes']):
        text = str(line).replace('\n', ' ').strip()
        if not text or not box or len(box) < 4:
            continue
        nb = normalize_bbox([int(box[0]), int(box[1]), int(box[2]), int(box[3])], int(w), int(h))
        for word in text.split():
            toks.append(word); boxes.append(nb)
    return toks, boxes


## 5. Processor & feature preparation


In [ ]:
from transformers import AutoProcessor
model_id = 'microsoft/layoutlmv3-base'
processor = AutoProcessor.from_pretrained(model_id, apply_ocr=False)

def prepare_examples(examples):
    images, words, boxes, labels = [], [], [], []
    for i in range(len(examples['image'])):
        row = {'words': examples['words'][i], 'bboxes': examples['bboxes'][i],
               'entities': examples['entities'][i],
               'image_size': examples['image_size'][i] if 'image_size' in examples else None,
               'image': examples['image'][i]}
        toks, bxs = row_to_words_boxes(row)
        gt = {k: (row['entities'].get(k) or '') for k in FIELDS}
        ids = [label2id[t] for t in tag_tokens(toks, gt)]
        images.append(examples['image'][i].convert('RGB'))
        words.append(toks); boxes.append(bxs); labels.append(ids)
    return processor(images=images, text=words, boxes=boxes, word_labels=labels,
                     truncation=True, padding='max_length', max_length=512)

# Quick sanity check that company/address now get labels
import collections
_chk = dataset['train'].select(range(8)).map(prepare_examples, batched=True, batch_size=4,
        remove_columns=dataset['train'].column_names)
_c = collections.Counter(int(x) for row in _chk['labels'] for x in row if x != -100)
print('label coverage on 8 docs:', {id2label[k]: _c[k] for k in sorted(_c)})
assert label2id['B-COMPANY'] in _c and label2id['B-ADDRESS'] in _c, 'company/address not labeled!'
print('OK: company and address are labeled.')


## 6. Model, metrics & training arguments


In [ ]:
import numpy as np, evaluate, torch
from transformers import (LayoutLMv3ForTokenClassification, TrainingArguments,
                          Trainer, default_data_collator)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    model_id, num_labels=len(label_list), id2label=id2label, label2id=label2id)

seqeval = evaluate.load('seqeval')
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    tp = [[label_list[pi] for pi, li in zip(pr, la) if li != -100] for pr, la in zip(preds, labels)]
    tl = [[label_list[li] for pi, li in zip(pr, la) if li != -100] for pr, la in zip(preds, labels)]
    r = seqeval.compute(predictions=tp, references=tl)
    out = {'precision': r['overall_precision'], 'recall': r['overall_recall'],
           'overall_f1': r['overall_f1'], 'accuracy': r['overall_accuracy']}
    for ent in ['COMPANY', 'DATE', 'ADDRESS', 'TOTAL']:
        if ent in r:
            out[ent.lower() + '_f1'] = r[ent]['f1']
    return out

args = TrainingArguments(
    output_dir='checkpoints', learning_rate=2e-5,
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=2, num_train_epochs=10,
    weight_decay=0.01, warmup_ratio=0.1, fp16=torch.cuda.is_available(),
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='overall_f1',
    greater_is_better=True, logging_steps=25, report_to=[])


## 7. Tokenize splits & train


In [ ]:
encoded = dataset.map(prepare_examples, batched=True, batch_size=4,
                      remove_columns=dataset['train'].column_names)
encoded.set_format('torch')

kw = dict(model=model, args=args, train_dataset=encoded['train'],
          eval_dataset=encoded['test'], data_collator=default_data_collator,
          compute_metrics=compute_metrics)
if 'processing_class' in Trainer.__init__.__code__.co_varnames:
    kw['processing_class'] = processor
else:
    kw['tokenizer'] = processor
trainer = Trainer(**kw)
trainer.train()


## 8. Final per-field evaluation


In [ ]:
metrics = trainer.evaluate()
print('Overall  F1:', round(metrics.get('eval_overall_f1', 0), 4))
for f in ['company', 'date', 'address', 'total']:
    print(f'{f:>8} F1:', round(metrics.get('eval_' + f + '_f1', 0), 4))


## 9. Save, zip & download the checkpoint


In [ ]:
import os, shutil
save_dir = 'best_layoutlmv3_sroie'
trainer.save_model(save_dir)
processor.save_pretrained(save_dir)   # also writes preprocessor_config.json
shutil.make_archive(save_dir, 'zip', save_dir)
print('Created', save_dir + '.zip', round(os.path.getsize(save_dir + '.zip')/1e6, 1), 'MB')
try:
    from google.colab import files
    files.download(save_dir + '.zip')
except Exception as e:
    print('Download from the Colab file browser instead:', e)


## 10. Next step (on your machine)
Unzip into `models/best_layoutlmv3_sroie/`, then re-run the honest evaluation:
```bash
python scripts/download_data.py --source hf --split test --output-dir data/test
python scripts/evaluate.py --data-dir data/test --ocr-engine sroie_precomputed
```
Send me the new numbers and I'll finalize the README.
